# arc-fs: adaptive resampling with pruned trees

Breiman's arc-fs (1998) is the version of AdaBoost behind the AdaBoost column of his random forests paper (2001, Table 2). Each step draws a training set from the cases with probabilities that favour the ones the earlier trees got wrong, grows a CART tree on it and prunes the tree by cost complexity on a second draw; the trees then vote with weights. No Julia package has arc-fs, and none prunes a CART tree by cost complexity on a test set. Both are written out here as documented functions, then tested against Breiman's published test errors on 20 data sets.

The trees are grown with DecisionTree.jl; arc-fs, the pruning and the weighted vote are a few lines each. The tests use 15 real data sets from PollisDatasets, four simulated ones generated as Breiman did, and the zip-code digits, downloaded only if you agree.

In [ ]:
Pollis.packages("DecisionTree", "StatsBase", "PollisDatasets", "Plots", "CodecZlib", "ProgressLogging")

## Set-up

Breiman's test errors are averages of 100 random runs, so `compare` prints each Julia average with its standard error, rounded beside the published value, and whether the two differ by less than two standard errors of a difference of two such averages (√2 se), Breiman's being one too. `repetitions` sets the number of runs: 100 as in the paper. Every run has its own seed, so the results do not depend on the number of threads; to use more, set **julia.NumThreads** to `auto` in the settings and restart the kernel.

`inputs` reads a data set as Breiman used it: the inputs as a matrix, a yes vote as 1, the classes numbered, and the few missing values replaced by the median of their input.

In [ ]:
using DecisionTree, StatsBase, PollisDatasets, Plots, CodecZlib, ProgressLogging, DelimitedFiles, Downloads, Printf, Random, Statistics
using DecisionTree: Leaf, Node

"""
	compare(title, rows)

Print each Julia error rate, in percent, with its standard error, rounded beside the published value, and whether they differ by less than
two standard errors of a difference (√2 se). Each row is `(label, value, se, published)`.
"""
function compare(title, rows)
	@printf("\n%s\n%-16s%10s%8s%10s%11s\n", title, "", "Julia", "se", "rounded", "published")
	for (label, value, se, published) in rows
		z = (value - parse(Float64, published)) / (sqrt(2) * se)
		@printf("%-16s%10.3f%8.3f%10.1f%11s  %s\n", label, value, se, value, published, abs(z) < 2 ? "within noise" : @sprintf("DIFFERENT (%.1f se)", z))
	end
end

repetitions = 100  # runs per data set, as in the paper
println("threads: ", Threads.nthreads())

"""Run `f(rng)` for runs 1:n, each with its own seed, on all threads, with a progress bar named `name`."""
function runs(f, n; name="runs")
	r = Vector{Any}(undef, n)
	finished = Threads.Atomic{Int}(0)
	@withprogress name=name begin
		Threads.@threads for i in 1:n
			r[i] = f(Xoshiro(i))
			@logprogress (Threads.atomic_add!(finished, 1) + 1) / n
		end
	end
	r
end

const CLASS = Dict("glass" => :type, "breastwisconsin" => :class, "pima" => :diabetes, "sonar" => :class, "vowel" => :vowel,
	"ionosphere" => :class, "vehicle" => :class, "soybean" => :disease, "germancredit" => :credit, "image" => :class,
	"ecoli" => :site, "votes" => :party, "liver" => :selector, "letters" => :letter, "satimage" => :class)
const NOTINPUT = [:id, :protein, :speaker, :segment, :set]  # identifiers and the train/test marker

"""The inputs of data set `id` as a matrix, missing values replaced by the median of their input, and the classes numbered 1, 2, …"""
function inputs(id)
	d = dataset(id)
	cols = [k for k in keys(d) if k ∉ NOTINPUT && k != CLASS[id]]
	X = reduce(hcat, [begin
		c = [v isa AbstractString ? Float64(v == "y") : v for v in d[k]]  # votes: yes = 1, no = 0
		m = median(skipmissing(c))
		Float64[ismissing(v) ? m : v for v in c]
	end for k in cols])
	labels = string.(d[CLASS[id]])
	names = sort(unique(labels))
	X, indexin(labels, names), length(names)
end

## The method

### Cost-complexity pruning

CART prunes a tree by **cost complexity** (Breiman et al. 1984, Ch. 3). It repeatedly collapses the weakest link, the internal node t with the smallest

g(t) = (R(t) − R(Tₜ)) / (|Tₜ| − 1),

where R(t) is the number of training cases misclassified at t as a leaf, R(Tₜ) the number misclassified by the branch below it, and |Tₜ| the leaves of the branch. This gives a sequence of ever smaller subtrees, down to the root; `prune` keeps the one with the fewest errors on a second sample, the smallest on ties. `classify` runs cases down a tree without recursion, since the deepest trees overflow the stack of DecisionTree.jl's recursive `apply_tree`.

In [ ]:
"""Run each row of `X` down tree `t`."""
function classify(t, X::AbstractMatrix)
	t isa DecisionTree.Root && (t = t.node)
	map(eachrow(X)) do x
		s = t
		while s isa Node
			s = x[s.featid] < s.featval ? s.left : s.right
		end
		s.majority
	end
end

caselabels(t::Leaf) = t.values  # the training classes that reached each leaf
caselabels(t::Node) = vcat(caselabels(t.left), caselabels(t.right))
misclassified(v) = length(v) - maximum(values(countmap(v)))  # R(t): errors as a leaf
nleaves(t::Leaf) = 1
nleaves(t::Node) = nleaves(t.left) + nleaves(t.right)
branchmisclassified(t::Leaf) = misclassified(t.values)
branchmisclassified(t::Node) = branchmisclassified(t.left) + branchmisclassified(t.right)

"""g(t): the training errors saved per leaf by the branch below internal node `t`."""
g(t::Node) = (misclassified(caselabels(t)) - branchmisclassified(t)) / (nleaves(t) - 1)

"""The smallest g(t) over the internal nodes of `t`: the weakest link."""
weakest(t::Leaf) = Inf
weakest(t::Node) = min(g(t), weakest(t.left), weakest(t.right))

"""Collapse to a leaf every internal node with g(t) at most `alpha`."""
cut(t::Leaf, alpha) = t
cut(t::Node, alpha) = g(t) <= alpha + 1e-12 ? (v = caselabels(t); Leaf(mode(v), v)) : Node(t.featid, t.featval, cut(t.left, alpha), cut(t.right, alpha))

"""
	prune(t, X, y)

The subtree of the cost-complexity sequence of tree `t` with the fewest errors on the cases (X, y), the smallest on ties.
"""
function prune(t, X, y)
	best, fewest = t, sum(classify(t, X) .!= y)
	while t isa Node
		t = cut(t, weakest(t))
		e = sum(classify(t, X) .!= y)
		if e <= fewest
			best, fewest = t, e
		end
	end
	best
end

### arc-fs

arc-fs (Breiman 1998, pp. 6-7) keeps a probability p(n) on each training case, equal at the start. At step k it draws a training set T′ of n cases with these probabilities, grows a CART tree on it, and prunes the tree on a second draw T″. With d(n) = 1 for the cases of the full training set the tree misclassifies,

εₖ = Σ p(n) d(n),  βₖ = (1 − εₖ) / εₖ,  p(n) ← p(n) βₖ^d(n), normalised,

so the misclassified cases weigh more in the next draw. The trees vote with weights log βₖ. When εₖ reaches ½, or is 0, the probabilities are set equal again and the construction restarts; the tree of that step does not vote.

In [ ]:
"""
	arcfs(X, y, K, rng)

Breiman's (1998) arc-fs: `K` steps of adaptive resampling with Freund and Schapire's update, each tree pruned on a second draw.
Returns the trees and their weights log β. A restart drops the tree of its step.
"""
function arcfs(X, y, K, rng)
	n = length(y)
	p = fill(1 / n, n)
	trees, weights = Any[], Float64[]
	for k in 1:K
		draw1, draw2 = sample(rng, 1:n, Weights(p), n), sample(rng, 1:n, Weights(p), n)
		full = build_tree(y[draw1], X[draw1, :], 0, -1, 1, 2, 0.0; loss=DecisionTree.util.gini, rng=rng, impurity_importance=false)
		t = prune(full.node, X[draw2, :], y[draw2])
		d = classify(t, X) .!= y
		ε = sum(p[d])
		if ε >= 1 / 2 || ε == 0
			fill!(p, 1 / n)  # restart
			continue
		end
		β = (1 - ε) / ε
		p .*= β .^ d
		p ./= sum(p)
		push!(trees, t)
		push!(weights, log(β))
	end
	trees, weights
end

"""The weighted vote of `trees` on each row of `X`, with `C` classes."""
function vote(trees, X, C; weights=ones(length(trees)))
	votes = zeros(size(X, 1), C)
	for (t, w) in zip(trees, weights)
		for (i, c) in enumerate(classify(t, X))
			votes[i, c] += w
		end
	end
	[argmax(r) for r in eachrow(votes)]
end

"""The test error, in percent, of `K` steps of arc-fs trained on the cases `train` and tested on `test`."""
function testerror(X, y, C, train, test, rng; K=50)
	trees, weights = arcfs(X[train, :], y[train], K, rng)
	100mean(vote(trees, X[test, :], C; weights=weights) .!= y[test])
end

## Tests

The published values are the AdaBoost column of Breiman (2001, Table 2, p. 11), which are arc-fs's: those for breast cancer, ionosphere, diabetes, glass, letters, sat-images and zip-code are also in the arcing paper's Table 5 (Breiman 1998), and those for waveform and twonorm in its Table 3. Soybean's is in the forests paper's Section 6 (p. 15).

### The small data sets

On each small data set a random 10% is set aside as the test set and arc-fs combines 50 trees grown on the rest; the test errors of 100 such runs are averaged (Breiman 2001, pp. 10-11).

In [ ]:
small = ["glass", "breastwisconsin", "pima", "sonar", "vowel", "ionosphere", "vehicle", "soybean", "germancredit", "image", "ecoli", "votes", "liver"]
published = Dict("glass" => "22.0", "breastwisconsin" => "3.2", "pima" => "26.6", "sonar" => "15.6", "vowel" => "4.1", "ionosphere" => "6.4",
	"vehicle" => "23.2", "soybean" => "5.8", "germancredit" => "23.5", "image" => "1.6", "ecoli" => "14.8", "votes" => "4.8", "liver" => "30.7",
	"letters" => "3.4", "satimage" => "8.8", "zip" => "6.2", "waveform" => "17.8", "twonorm" => "4.9", "threenorm" => "18.8", "ringnorm" => "6.9")
results = Dict{String, Tuple{Float64, Float64}}()  # data set => (average test error, standard error)

for id in small
	X, y, C = inputs(id)
	r = Float64.(runs(repetitions; name=id) do rng
		n = length(y)
		test = randperm(rng, n)[1:round(Int, n / 10)]
		testerror(X, y, C, setdiff(1:n, test), test, rng)
	end)
	results[id] = (mean(r), std(r) / sqrt(length(r)))
end
compare("arc-fs, 50 trees, $repetitions runs of a 10% test set (Breiman 2001, Table 2 and p. 15)", [(id, results[id]..., published[id]) for id in small])

### The larger data sets

Letters, sat-images and zip-code have their own training and test sets, so each is a single run: 50 trees, 100 for zip-code. With one run there is no average; the standard error shown is the binomial one of a test error on that many cases, sqrt(e(100 − e)/n). arc-fs grows and prunes every tree on all the inputs, so letters takes several minutes.

The zip-code digits, 9,298 handwritten digits of 16 × 16 grey levels, are not in PollisDatasets. Set `download_zipcode = true` to download them, 2.3 MB, from the site of Hastie, Tibshirani and Friedman's *The Elements of Statistical Learning* into ~/.pollis/data/zipcode; otherwise the zip-code row is left out.

In [ ]:
download_zipcode = false  # set to true to download the zip-code data (2.3 MB, once)
zipdir = joinpath(homedir(), ".pollis", "data", "zipcode")
if download_zipcode
	mkpath(zipdir)
	for f in ("zip.train.gz", "zip.test.gz")
		isfile(joinpath(zipdir, f)) || Downloads.download("https://hastie.su.domains/ElemStatLearn/datasets/$f", joinpath(zipdir, f))
	end
end

"""A zip-code file: the digit, then the 256 grey levels, one case per line."""
function zipread(f)
	d = readdlm(GzipDecompressorStream(open(f)), Float64)[:, 1:257]
	d[:, 2:end], Int.(d[:, 1]) .+ 1
end

"""One run on a data set with its own test set; the standard error is binomial."""
function largerun(id, X, y, C, train, test; K=50)
	e = @withprogress name=id testerror(X, y, C, train, test, Xoshiro(1998); K=K)
	results[id] = (e, sqrt(e * (100 - e) / length(test)))
end

X, y, C = inputs("letters")
largerun("letters", X, y, C, 1:15000, 15001:20000)  # the first 15,000 cases train: the papers do not say which
X, y, C = inputs("satimage")
set = dataset("satimage").set
largerun("satimage", X, y, C, findall(==("train"), set), findall(==("test"), set))
large = ["letters", "satimage"]
if isfile(joinpath(zipdir, "zip.test.gz"))
	X1, y1 = zipread(joinpath(zipdir, "zip.train.gz"))
	X2, y2 = zipread(joinpath(zipdir, "zip.test.gz"))
	largerun("zip", vcat(X1, X2), vcat(y1, y2), 10, 1:length(y1), length(y1)+1:length(y1)+length(y2); K=100)
	push!(large, "zip")
end
compare("arc-fs, one run on the given test set (Breiman 2001, Table 2)", [(id, results[id]..., published[id]) for id in large])

### The simulated data sets

The four simulated data sets are those of the arcing paper (Breiman 1998, p. 7), with two classes, or three for waveform, equally likely:

- **twonorm**: 20 inputs, normal with identity covariance and mean (a, …, a) for class 1, (−a, …, −a) for class 2, a = 2/√20.
- **threenorm**: class 1 an equal mixture of the normals with means (a, …, a) and (−a, …, −a), class 2 the normal with mean (a, −a, a, −a, …), a = 2/√20.
- **ringnorm**: class 1 normal with mean 0 and covariance 4 times the identity, class 2 normal with mean (a, …, a) and the identity, a = 1/√20.
- **waveform** (Breiman et al. 1984, p. 49): 21 inputs, each class a random mixture u h + (1 − u) h′ of two of three triangular waves plus N(0, 1) noise: class 1 mixes h₁ and h₂, class 2 h₁ and h₃, class 3 h₂ and h₃, with u uniform on (0, 1), h₁(i) = max(6 − |i − 11|, 0), h₂(i) = h₁(i − 4) and h₃(i) = h₁(i + 4).

Each of 50 runs draws a training set of 300 cases and a test set of 3,000 (Breiman 2001, p. 11).

In [ ]:
function twonorm(n, rng)
	a = 2 / sqrt(20)
	y = rand(rng, 1:2, n)
	randn(rng, n, 20) .+ a .* ifelse.(y .== 1, 1, -1), y
end

function threenorm(n, rng)
	a = 2 / sqrt(20)
	y = rand(rng, 1:2, n)
	alternating = [isodd(j) ? a : -a for j in 1:20]
	X = randn(rng, n, 20)
	for i in 1:n
		X[i, :] .+= y[i] == 2 ? alternating : fill(rand(rng, Bool) ? a : -a, 20)
	end
	X, y
end

function ringnorm(n, rng)
	a = 1 / sqrt(20)
	y = rand(rng, 1:2, n)
	X = randn(rng, n, 20)
	ifelse.(y .== 1, 2 .* X, X .+ a), y
end

function waveform(n, rng)
	h1 = [max(6 - abs(i - 11), 0) for i in 1:21]
	h2 = [max(6 - abs(i - 4 - 11), 0) for i in 1:21]
	h3 = [max(6 - abs(i + 4 - 11), 0) for i in 1:21]
	mixtures = ((h1, h2), (h1, h3), (h2, h3))
	y = rand(rng, 1:3, n)
	X = randn(rng, n, 21)
	for i in 1:n
		u = rand(rng)
		h, h′ = mixtures[y[i]]
		X[i, :] .+= u .* h .+ (1 - u) .* h′
	end
	X, y
end

simulated = ["waveform", "twonorm", "threenorm", "ringnorm"]
for (id, simulate, C) in zip(simulated, (waveform, twonorm, threenorm, ringnorm), (3, 2, 2, 2))
	r = Float64.(runs(50; name=id) do rng
		X, y = simulate(300, rng)
		Xt, yt = simulate(3000, rng)
		testerror([X; Xt], [y; yt], C, 1:300, 301:3300, rng)
	end)
	results[id] = (mean(r), std(r) / sqrt(length(r)))
end
compare("arc-fs, 50 trees, 50 runs of 300 training and 3,000 test cases (Breiman 2001, Table 2)", [(id, results[id]..., published[id]) for id in simulated])

All the tests at once: each published test error against Julia's, with two standard errors of a difference either way.

In [ ]:
ids = sort(collect(keys(results)); by=id -> parse(Float64, published[id]))
p, j, s = [parse(Float64, published[id]) for id in ids], [results[id][1] for id in ids], [results[id][2] for id in ids]
scatter(p, j, yerror=2sqrt(2) .* s, xlabel="Breiman, test error %", ylabel="Julia, test error %", label="", size=(520, 500), aspect_ratio=:equal, markersize=4)
plot!([0, 32], [0, 32], color=:gray, linestyle=:dash, label="")

## Example

On one split of the breast cancer data, the test error of arc-fs as trees are added, against a single CART tree pruned the same way: the vote of a few dozen pruned trees halves the error of one.

In [ ]:
X, y, C = inputs("breastwisconsin")
rng = Xoshiro(7)
test = randperm(rng, length(y))[1:70]
train = setdiff(1:length(y), test)
trees, weights = arcfs(X[train, :], y[train], 50, rng)
curve = [100mean(vote(trees[1:k], X[test, :], C; weights=weights[1:k]) .!= y[test]) for k in 1:length(trees)]
draw1, draw2 = sample(rng, train, length(train)), sample(rng, train, length(train))
full = build_tree(y[draw1], X[draw1, :], 0, -1, 1, 2, 0.0; loss=DecisionTree.util.gini, rng=rng, impurity_importance=false)
one = prune(full.node, X[draw2, :], y[draw2])
plot(curve, xlabel="trees", ylabel="test error %", label="arc-fs", size=(560, 330))
hline!([100mean(classify(one, X[test, :]) .!= y[test])], label="one pruned tree", linestyle=:dash)

## What differs

In one run of this notebook, 18 of the 20 test errors agreed with Breiman's within noise. Breast cancer (3.7 against 3.2) and diabetes (25.3 against 26.6) came closest to the limit. The two that did not agree:

- **Twonorm (4.6 against 4.9) and ringnorm (6.2 against 6.9)**, below Breiman by 2.8 and 4.4 standard errors. The generators follow the arcing paper's description (p. 7). The forests of the Out-of-bag notebook show the same small gap on these two sets, so the difference may be in how the trees are grown rather than in arc-fs. Not explained.

arc-fs also depends on details the arcing paper leaves open: whether the tree of a step that restarts joins the vote (here it does not), and how its CART chose among equally good pruned subtrees (here the smallest).

**The data.** Soybean has 683 cases, not 685, and its categorical inputs are split as ordered codes, since DecisionTree.jl has no splits on subsets of categories. Missing values are replaced by medians; Breiman does not say what he did.

## References

- Breiman, L. (1998). Arcing classifiers. *The Annals of Statistics* 26(3), 801-849. doi:10.1214/aos/1024691079
- Breiman, L. (2001). Random forests. *Machine Learning* 45(1), 5-32. doi:10.1023/A:1010933404324
- Breiman, L., Friedman, J. H., Olshen, R. A. and Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.
- Freund, Y. and Schapire, R. E. (1996). Experiments with a new boosting algorithm. *Machine Learning: Proceedings of the Thirteenth International Conference*, 148-156.
- Hastie, T., Tibshirani, R. and Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed. Springer: the zip-code data.